# A11 — Pipeline End-to-End: Modelo Final

**Objetivo:** Consolidar todo o projeto em um pipeline integrado e reprodutível,
conectando preparação de dados, treinamento, avaliação, inferência e geração
automática de resultados.

## Estrutura do pipeline

| Etapa | Módulo | Descrição |
|-------|--------|-----------|
| 1. Setup | `config.yaml` + `reprodutibilidade` | Seed global, caminhos, hiperparâmetros |
| 2. Pré-processamento | `src.preprocessing` | CSV -> tensor 4D, split por image_id, tf.data |
| 3. Treinamento | `src.training` | MobileNetV2 em 2 fases (head + fine-tuning) |
| 4. Avaliação | `src.evaluation` | Métricas, curvas ROC/PR, matrizes de confusão |
| 5. Inferência | `src.inference` | Predição em lote, exportação padronizada |

**Execução simples:** basta executar todas as células em sequência (`Run All`).

## 1. Setup — Reprodutibilidade e configuração

In [1]:
"""
Célula 1 — Configuração do ambiente e seed global.

Garante que todos os experimentos sejam reprodutíveis fixando seeds
em Python, NumPy e TensorFlow, além de ativar determinismo de operações.
"""
import json
import logging
import sys
import warnings
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    confusion_matrix,
)
from tensorflow import keras

warnings.filterwarnings('ignore')


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'artefatos' / 'a11_pipeline_e2e' / 'config.yaml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto a partir do diretorio atual.')


PROJECT_ROOT = find_project_root()
ARTIFACT_ROOT = PROJECT_ROOT / 'artefatos' / 'a11_pipeline_e2e'
NOTEBOOK_DIR = ARTIFACT_ROOT / 'notebooks'
CONFIG_PATH = ARTIFACT_ROOT / 'config.yaml'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('a11_pipeline')

from src.inference.keras_binary_predict import collect_binary_predictions
from src.models.cnn_data_prep import prepare_grouped_cnn_splits
from src.models.transfer_experiment_runner import TransferLearningExperimentRunner
from src.reprodutibilidade import set_global_seed

logger.info('Raiz do projeto: %s', PROJECT_ROOT)
logger.info('Raiz do artefato: %s', ARTIFACT_ROOT)
logger.info('Notebook: %s', NOTEBOOK_DIR / 'a11_pipeline_e2e.ipynb')


08:00:47 [INFO] a11_pipeline - Raiz do projeto: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01
08:00:47 [INFO] a11_pipeline - Raiz do artefato: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e
08:00:47 [INFO] a11_pipeline - Notebook: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/notebooks/a11_pipeline_e2e.ipynb


In [2]:
"""
Célula 2 — Carregamento do config e preparação das pastas de saída.

Todos os caminhos relativos do A11 são resolvidos a partir do config.yaml,
mantendo o artefato autocontido e reprodutível.
"""
def resolve_path(base_dir: Path, path_str: str) -> Path:
    path = Path(path_str)
    if path.is_absolute():
        return path
    return (base_dir / path).resolve()


def load_config(config_path: str | Path) -> dict[str, Any]:
    config_path = Path(config_path).resolve()
    with config_path.open('r', encoding='utf-8') as file:
        raw_config = yaml.safe_load(file)

    config = deepcopy(raw_config)
    config['_config_path'] = config_path
    config['_config_dir'] = config_path.parent
    for key, value in config['paths'].items():
        config['paths'][key] = resolve_path(config_path.parent, value)
    return config


cfg = load_config(CONFIG_PATH)
set_global_seed(int(cfg['seed']))

DATASET_CSV = cfg['paths']['dataset_csv']
EXTRACTED_CODES_JSON = cfg['paths']['extracted_codes_json']

assert DATASET_CSV.exists(), f'Dataset nao encontrado: {DATASET_CSV}'
assert EXTRACTED_CODES_JSON.exists(), f'Codigos nao encontrados: {EXTRACTED_CODES_JSON}'

OUTPUT_MODELS = cfg['paths']['outputs_models']
OUTPUT_METRICS = cfg['paths']['outputs_metrics']
OUTPUT_VIZ = cfg['paths']['outputs_viz']
OUTPUT_PREDS = cfg['paths']['outputs_preds']

for directory in (OUTPUT_MODELS, OUTPUT_METRICS, OUTPUT_VIZ, OUTPUT_PREDS):
    directory.mkdir(parents=True, exist_ok=True)
(OUTPUT_MODELS / 'runs').mkdir(parents=True, exist_ok=True)

LIMIT_SAMPLES = None
SKIP_TRAIN = False
SKIP_INFERENCE = False

logger.info('Seed: %d', int(cfg['seed']))
logger.info('Config carregada: %s', CONFIG_PATH)
logger.info('Dataset: %s', DATASET_CSV)
logger.info('Codigos: %s', EXTRACTED_CODES_JSON)
logger.info('Saidas -> modelos: %s', OUTPUT_MODELS)
logger.info('Saidas -> metricas: %s', OUTPUT_METRICS)
logger.info('Saidas -> visualizacoes: %s', OUTPUT_VIZ)
logger.info('Saidas -> predicoes: %s', OUTPUT_PREDS)
print('\n✔ Setup concluido com sucesso.')


08:00:53 [INFO] a11_pipeline - Seed: 42
08:00:53 [INFO] a11_pipeline - Config carregada: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/config.yaml
08:00:53 [INFO] a11_pipeline - Dataset: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/data/pixels_dataset.csv
08:00:53 [INFO] a11_pipeline - Codigos: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/data/extracted_codes.json
08:00:53 [INFO] a11_pipeline - Saidas -> modelos: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models
08:00:53 [INFO] a11_pipeline - Saidas -> metricas: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/metrics
08:00:53 [INFO] a11_pipeline - Saidas -> visualizacoes: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/visualizations
08:00:53 [INFO] a11_pipeline - Saidas -> predicoes: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs

[Reprodutibilidade] Seed definida: 42

✔ Setup concluido com sucesso.


## 2. Pré-processamento — Dados → Tensores → tf.data

Etapas automatizadas:
1. Carrega `pixels_dataset.csv` e extrai labels de `extracted_codes.json`.
2. Converte colunas `pixel_*` em tensor 4D `(N, H, W, C)`.
3. Split estratificado por `image_id` (treino / validação / teste) para evitar vazamento de dados.
4. Normalização z-score por canal (ajustada apenas no treino).
5. Data augmentation no treino (flip, rotação, contraste).

In [3]:
"""
Célula 3 — Funções de pré-processamento.

A lógica fica explícita no notebook para atender à entrega, mas reaproveita
os componentes centrais do projeto para manter consistência com o pipeline oficial.
"""
def run_preprocessing(
    cfg: dict[str, Any],
    dataset_csv: str | Path,
    extracted_codes_json: str | Path,
    limit_samples: int | None = None,
) -> dict[str, Any]:
    set_global_seed(int(cfg['seed']))
    df = pd.read_csv(dataset_csv)
    if limit_samples is not None:
        df = df.sample(n=min(limit_samples, len(df)), random_state=int(cfg['seed']))

    splits = prepare_grouped_cnn_splits(
        df,
        extracted_codes_path=str(extracted_codes_json),
        test_size=float(cfg['data']['test_size']),
        val_size=float(cfg['data']['val_size']),
        seed=int(cfg['seed']),
    )

    runner = TransferLearningExperimentRunner(cfg['model']['base_config_name'])
    runner.config = {
        'model': {
            'input_shape': [
                int(cfg['data']['image_size'][0]),
                int(cfg['data']['image_size'][1]),
                int(cfg['data']['num_bands']),
            ],
            'num_classes': int(cfg['model']['num_classes']),
            'backbone': cfg['model']['backbone'],
            'resize_to': [int(v) for v in cfg['model']['resize_to']],
            'dropout_rate': float(cfg['model']['dropout_rate']),
            'fine_tune_last_layers': int(cfg['model']['fine_tune_last_layers']),
        },
        'training': {
            'batch_size': int(cfg['training']['batch_size']),
            'head_epochs': int(cfg['training']['head_epochs']),
            'head_learning_rate': float(cfg['training']['head_learning_rate']),
            'fine_tune_epochs': int(cfg['training']['fine_tune_epochs']),
            'fine_tune_learning_rate': float(cfg['training']['fine_tune_learning_rate']),
            'early_stopping_patience_head': int(cfg['training']['early_stopping_patience_head']),
            'early_stopping_patience_ft': int(cfg['training']['early_stopping_patience_ft']),
        },
        'data': {
            'dataset_path': str(dataset_csv),
            'codes_path': str(extracted_codes_json),
            'normalization_method': cfg['data']['normalization_method'],
            'test_size': float(cfg['data']['test_size']),
            'val_size': float(cfg['data']['val_size']),
        },
        'output': {
            'models_dir': str(OUTPUT_MODELS / 'runs'),
            'logs_dir': str(OUTPUT_METRICS),
            'save_model': False,
            'save_history': False,
        },
    }
    tf_data = runner.build_tf_data(splits)

    return {
        'splits': splits,
        'tf_data': tf_data,
        'runner': runner,
    }


In [4]:
prep_result = run_preprocessing(
    cfg,
    dataset_csv=DATASET_CSV,
    extracted_codes_json=EXTRACTED_CODES_JSON,
    limit_samples=LIMIT_SAMPLES,
)

splits = prep_result['splits']
tf_data = prep_result['tf_data']

print('\n' + '═' * 60)
print('RESUMO DO PRÉ-PROCESSAMENTO')
print('═' * 60)
print(f"  Amostras treino:     {splits['X_train'].shape[0]:>6d}  |  shape: {splits['X_train'].shape}")
print(f"  Amostras validação:  {splits['X_val'].shape[0]:>6d}  |  shape: {splits['X_val'].shape}")
print(f"  Amostras teste:      {splits['X_test'].shape[0]:>6d}  |  shape: {splits['X_test'].shape}")
print(f"  Input shape (CNN):   {tf_data['train_meta']['input_shape']}")
print(f"  Normalização:        {tf_data['train_meta']['normalization']}")
print(f"  Augmentação (treino):{tf_data['train_meta']['augment']}")
print('═' * 60)
print('\n✔ Pré-processamento concluído.')


[Reprodutibilidade] Seed definida: 42
TransferLearningExperimentRunner inicializado: tl_baseline

════════════════════════════════════════════════════════════
RESUMO DO PRÉ-PROCESSAMENTO
════════════════════════════════════════════════════════════
  Amostras treino:        177  |  shape: (177, 128, 128, 9)
  Amostras validação:      59  |  shape: (59, 128, 128, 9)
  Amostras teste:          59  |  shape: (59, 128, 128, 9)
  Input shape (CNN):   (160, 160, 9)
  Normalização:        zscore
  Augmentação (treino):True
════════════════════════════════════════════════════════════

✔ Pré-processamento concluído.


## 3. Treinamento — MobileNetV2 via Transfer Learning

O modelo final oficial do A11 é treinado em duas fases:
1. Treino da cabeça classificadora congelando o backbone.
2. Fine-tuning das últimas camadas do backbone MobileNetV2.


In [5]:
"""
Célula 4 — Funções de treinamento e avaliação em teste.
"""
def save_history(history_path: Path, history: dict[str, list[float]]) -> None:
    serializable = {
        key: [float(value) for value in values]
        for key, values in history.items()
    }
    with history_path.open('w', encoding='utf-8') as file:
        json.dump(serializable, file, indent=2, ensure_ascii=False)


def evaluate_loaded_model(
    runner: TransferLearningExperimentRunner,
    tf_data: dict[str, Any],
    cfg: dict[str, Any],
    model_path: Path,
    history_path: Path,
) -> dict[str, Any]:
    metrics = runner.evaluate_on_test(tf_data)
    total_epochs = None
    if history_path.exists():
        with history_path.open('r', encoding='utf-8') as file:
            history = json.load(file)
        total_epochs = len(history.get('loss', [])) or None

    return {
        'config_name': cfg['model']['base_config_name'],
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'experiment_dir': str(model_path.parent),
        'training_time_seconds': None,
        'head_epochs': None,
        'ft_epochs': None,
        'total_epochs': total_epochs,
        'final_train_loss': None,
        'final_train_acc': None,
        'final_val_loss': None,
        'final_val_acc': None,
        **metrics,
    }


def run_training(
    cfg: dict[str, Any],
    prep_result: dict[str, Any],
    skip_train: bool = False,
) -> dict[str, Any]:
    set_global_seed(int(cfg['seed']))

    runner = prep_result['runner']
    tf_data = prep_result['tf_data']
    model_path = OUTPUT_MODELS / 'best_model.keras'
    history_path = OUTPUT_MODELS / 'history.json'

    if skip_train and not model_path.exists():
        raise FileNotFoundError(f'--skip-train exige modelo salvo em: {model_path}')

    if skip_train:
        runner.model = keras.models.load_model(model_path)
        result = evaluate_loaded_model(runner, tf_data, cfg, model_path, history_path)
        return {
            'runner': runner,
            'tf_data': tf_data,
            'result': result,
            'model_path': model_path,
            'history_path': history_path if history_path.exists() else None,
        }

    runner.create_experiment_dir()
    runner.build_model(input_shape=tf_data['train_meta']['input_shape'])
    train_result = runner.train_two_phases(tf_data, verbose=0)
    test_metrics = runner.evaluate_on_test(tf_data)

    runner.model.save(model_path)
    save_history(history_path, train_result['full_history'])

    result = {
        'config_name': cfg['model']['base_config_name'],
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'experiment_dir': str(runner.experiment_dir),
        'training_time_seconds': runner.training_time,
        'head_epochs': train_result['head_epochs'],
        'ft_epochs': train_result['ft_epochs'],
        'total_epochs': train_result['total_epochs'],
        'final_train_loss': float(train_result['full_history']['loss'][-1]),
        'final_train_acc': float(train_result['full_history']['accuracy'][-1]),
        'final_val_loss': float(train_result['full_history']['val_loss'][-1]),
        'final_val_acc': float(train_result['full_history']['val_accuracy'][-1]),
        **test_metrics,
    }

    return {
        'runner': runner,
        'tf_data': tf_data,
        'result': result,
        'model_path': model_path,
        'history_path': history_path,
    }


In [6]:
execution = run_training(
    cfg,
    prep_result=prep_result,
    skip_train=SKIP_TRAIN,
)

result = execution['result']

print('\n' + '═' * 60)
print('RESUMO DO TREINAMENTO E AVALIAÇÃO')
print('═' * 60)
print(f"  Test accuracy:        {result.get('test_accuracy')}")
print(f"  Test precision:       {result.get('test_precision')}")
print(f"  Test recall:          {result.get('test_recall')}")
print(f"  Test F1-score:        {result.get('test_f1')}")
print(f"  Test balanced acc.:   {result.get('test_balanced_accuracy')}")
print(f"  Test ROC AUC:         {result.get('test_roc_auc')}")
print(f"  Test PR AUC:          {result.get('test_pr_auc')}")
print(f"  Modelo salvo em:      {execution['model_path']}")
print(f"  Histórico salvo em:   {execution['history_path']}")
print('═' * 60)
print('\n✔ Treinamento concluído.')


[Reprodutibilidade] Seed definida: 42

Experimento: tl_baseline_20260408_080231

Construindo modelo Transfer Learning...
  Backbone: MobileNetV2
  Dropout: 0.25

--- Fase 1: Head Training ---
  LR: 0.0001
  Epochs: 6


E0000 00:00:1775646152.576304 3103129 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



--- Fase 2: Fine-Tuning ---
  LR: 1e-05
  Epochs: 12
  Camadas descongeladas: 20

  Head: 6 ep | FT: 7 ep | Total: 13 ep
  Tempo: 94.4s

Avaliando no conjunto de teste...
  Acc: 0.8814  F1: 0.8511  AUC: 0.9166666666666666

════════════════════════════════════════════════════════════
RESUMO DO TREINAMENTO E AVALIAÇÃO
════════════════════════════════════════════════════════════
  Accuracy:             0.8813559322033898
  Precision:            0.8333333333333334
  Recall:               0.8695652173913043
  F1-score:             0.851063829787234
  Balanced accuracy:    0.8792270531400965
  ROC AUC:              0.9166666666666666
  PR AUC:               0.8588168652945789
  Modelo salvo em:      /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models/best_model.keras
  Histórico salvo em:   /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models/history.json
══════════════════════════════════════════════════════

## 4. Inferência — Predições do conjunto de teste

Nesta etapa, o modelo final gera probabilidades e classes previstas no conjunto de teste,
exportando um arquivo padronizado com `image_id`, `y_true`, `y_pred` e `y_score`.


In [7]:
"""
Célula 5 — Função de inferência e exportação de predições.
"""
def run_inference(
    cfg: dict[str, Any],
    execution: dict[str, Any],
    image_ids_test,
) -> tuple[pd.DataFrame, Path]:
    predictions = collect_binary_predictions(
        model=execution['runner'].model,
        dataset=execution['tf_data']['test_ds'],
        sample_ids=[str(image_id) for image_id in image_ids_test],
    )
    predictions = predictions.rename(
        columns={
            'sample_id': 'image_id',
            'prob_pos': 'y_score',
        }
    )
    predictions['y_true'] = predictions['y_true'].astype(int)
    predictions['y_pred'] = (
        predictions['y_score'] >= float(cfg['evaluation']['threshold_default'])
    ).astype(int)
    predictions['split'] = 'test'

    predictions_path = OUTPUT_PREDS / 'test_predictions.csv'
    predictions[['image_id', 'y_true', 'y_pred', 'y_score', 'split']].to_csv(predictions_path, index=False)
    return predictions[['image_id', 'y_true', 'y_pred', 'y_score', 'split']], predictions_path


In [8]:
predictions_df = None
predictions_path = None

if not SKIP_INFERENCE:
    predictions_df, predictions_path = run_inference(
        cfg,
        execution=execution,
        image_ids_test=splits['image_ids_test'],
    )
    print('\n' + '═' * 60)
    print('PREDIÇÕES DE TESTE')
    print('═' * 60)
    print(predictions_df.head())
    print(f'\nArquivo exportado: {predictions_path}')
    print('═' * 60)
    print('\n✔ Inferência concluída.')
else:
    print('Inferência pulada por SKIP_INFERENCE=True.')



════════════════════════════════════════════════════════════
PREDIÇÕES DE TESTE
════════════════════════════════════════════════════════════
  image_id  y_true  y_pred   y_score split
0    23273       0       0  0.263306  test
1    23350       0       0  0.048475  test
2   23576B       0       0  0.261377  test
3    23577       0       1  0.597861  test
4    25636       1       1  0.601426  test

Arquivo exportado: /Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/predictions/test_predictions.csv
════════════════════════════════════════════════════════════

✔ Inferência concluída.


## 5. Avaliação final e exportação dos artefatos

A avaliação final consolida métricas de teste, gera visualizações e salva um resumo estruturado
do experimento em `.json` e `.csv`.


In [9]:
"""
Célula 6 — Funções de visualização e consolidação do experimento.
"""
def save_visualizations(predictions_df: pd.DataFrame) -> tuple[Path, Path]:
    confusion_matrix_path = OUTPUT_VIZ / 'confusion_matrix.png'
    roc_pr_path = OUTPUT_VIZ / 'roc_pr_curves.png'

    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(predictions_df['y_true'], predictions_df['y_pred'])
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Negativo', 'Positivo'],
    ).plot(ax=ax, colorbar=False)
    ax.set_title('Matriz de Confusao - Teste')
    fig.tight_layout()
    fig.savefig(confusion_matrix_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    y_true = predictions_df['y_true'].to_numpy()
    y_score = predictions_df['y_score'].to_numpy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    RocCurveDisplay.from_predictions(y_true, y_score, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=axes[1])
    axes[0].set_title('ROC - Teste')
    axes[1].set_title('Precision-Recall - Teste')
    fig.tight_layout()
    fig.savefig(roc_pr_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return confusion_matrix_path, roc_pr_path


def build_summary(
    cfg: dict[str, Any],
    splits: dict[str, Any],
    result: dict[str, Any],
    model_path: Path,
    history_path: Path | None,
    predictions_path: Path | None,
) -> dict[str, Any]:
    split_meta = splits['split_meta']
    return {
        'config_used': str(cfg['_config_path']),
        'timestamp': result['timestamp'],
        'seed': int(cfg['seed']),
        'model_name': cfg['model']['base_config_name'],
        'evaluation_split': result.get('evaluation_split', 'test'),
        'threshold': float(cfg['evaluation']['threshold_default']),
        'n_total': int(split_meta['n_total']),
        'n_valid': int(split_meta['n_valid']),
        'n_train': int(split_meta['n_train']),
        'n_val': int(split_meta['n_val']),
        'n_test': int(split_meta['n_test']),
        'test_accuracy': result.get('test_accuracy'),
        'test_precision': result.get('test_precision'),
        'test_recall': result.get('test_recall'),
        'test_f1': result.get('test_f1'),
        'test_balanced_accuracy': result.get('test_balanced_accuracy'),
        'test_roc_auc': result.get('test_roc_auc'),
        'test_pr_auc': result.get('test_pr_auc'),
        'training_time_seconds': result.get('training_time_seconds'),
        'total_epochs': result.get('total_epochs'),
        'head_epochs': result.get('head_epochs'),
        'ft_epochs': result.get('ft_epochs'),
        'model_path': str(model_path),
        'history_path': str(history_path) if history_path is not None else None,
        'predictions_path': str(predictions_path) if predictions_path is not None else None,
        'experiment_dir': result.get('experiment_dir'),
    }


def save_summary(summary: dict[str, Any]) -> tuple[Path, Path]:
    summary_json = OUTPUT_METRICS / 'summary.json'
    summary_csv = OUTPUT_METRICS / 'summary.csv'
    with summary_json.open('w', encoding='utf-8') as file:
        json.dump(summary, file, indent=2, ensure_ascii=False)
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)
    return summary_json, summary_csv


In [10]:
viz_paths = None
if predictions_df is not None:
    viz_paths = save_visualizations(predictions_df)

summary = build_summary(
    cfg,
    splits=splits,
    result=result,
    model_path=execution['model_path'],
    history_path=execution['history_path'],
    predictions_path=predictions_path,
)
summary_json_path, summary_csv_path = save_summary(summary)

print('\n' + '═' * 60)
print('RESUMO CONSOLIDADO DO EXPERIMENTO')
print('═' * 60)
print(json.dumps(summary, indent=2, ensure_ascii=False))
if viz_paths is not None:
    print(f'\nMatriz de confusao: {viz_paths[0]}')
    print(f'ROC/PR curves:      {viz_paths[1]}')
print(f'Summary JSON:        {summary_json_path}')
print(f'Summary CSV:         {summary_csv_path}')
print('═' * 60)
print('\n✔ Exportação final concluída.')



════════════════════════════════════════════════════════════
RESUMO CONSOLIDADO DO EXPERIMENTO
════════════════════════════════════════════════════════════
{
  "config_used": "/Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/config.yaml",
  "timestamp": "2026-04-08 08:04:10",
  "seed": 42,
  "model_name": "tl_baseline",
  "evaluation_split": "test",
  "threshold": 0.5,
  "n_total": 295,
  "n_valid": 295,
  "n_train": 177,
  "n_val": 59,
  "n_test": 59,
  "test_accuracy": 0.8813559322033898,
  "test_precision": 0.8333333333333334,
  "test_recall": 0.8695652173913043,
  "test_f1": 0.851063829787234,
  "test_balanced_accuracy": 0.8792270531400965,
  "test_roc_auc": 0.9166666666666666,
  "test_pr_auc": 0.8588168652945789,
  "training_time_seconds": 94.41106700897217,
  "total_epochs": 13,
  "head_epochs": 6,
  "ft_epochs": 7,
  "model_path": "/Users/giovanna-britto/Documents/INTELI/PROJETO/g01/artefatos/a11_pipeline_e2e/outputs/models/best_model.keras",
  "hi

## Artefatos gerados

Ao fim da execução completa, este notebook gera os arquivos padronizados do A11:

- `outputs/metrics/summary.json`
- `outputs/metrics/summary.csv`
- `outputs/models/best_model.keras`
- `outputs/models/history.json`
- `outputs/predictions/test_predictions.csv`
- `outputs/visualizations/confusion_matrix.png`
- `outputs/visualizations/roc_pr_curves.png`
